In [56]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px

from func import get_coefficient_df

import scipy.stats as stats
from matplotlib.pyplot import figure
from sklearn.linear_model import LogisticRegression

from sklearn.model_selection import train_test_split

from sklearn.svm import LinearSVC

from sklearn.preprocessing import MinMaxScaler


from sklearn.model_selection import GridSearchCV
from sklearn.metrics import make_scorer, f1_score, roc_auc_score, roc_curve, classification_report, confusion_matrix

## Goals
1) Load Dataset
2) Generate Target Values
3) Generate Demographic Features

In [57]:
target = "y_album_buying" # 
# target = "y_heal_b_kpop"

## 01) Load Dataset

In [58]:
df_clean = pd.read_csv('intermediateData/cleanData.csv', index_col=0)
print("Shape:", df_clean.shape)
df_clean.head(1)

Shape: (905, 559)


,Timestamp,What is your age group?,What is your gender?,What is your sexuality?,Where do you live?,What is your current occupation?,"Are you generally introverted, extroverted in real life?",How often do you browse r/Twice?,How did you discover r/Twice?,"Other than Twice, what other k-pop groups do you listen to?",...,x_ss_artm_mpp_memb_n_bit_2,x_ss_artm_mpp_memb_n_bit_3,x_xs_misc_pop_group_num,x_xs_misc_cvd_group_num,x_xs_misc_wop_group_num,x_xs_altm_pca_a1,x_xs_memb_pca_a1,x_xs_artm_pca_a1,x_xs_ms_clstr,x_xs_beha_evt_live_tour
0,12/26/2021 3:47:53,20-25,Female,Gay/Lesbian,North America,Unemployed,It depends who I'm around,A few times a month,Reddit Browsing,"BTS, EXO, ITZY, BlackPink, Red Velvet, SNSD, S...",...,1.0,0.0,1.0,0.0,0.0,0.52884,-0.02717,1.005668,0,1


#### Get Initial Feature and Target Counts

In [59]:
x_columns = [col for col in df_clean.columns if col.startswith('x_')] 
y_columns = [col for col in df_clean.columns if col.startswith('y_')]

def feature_size_count(column_list):
    feature_size_list = ["x_xs", "x_ss", "x_mm", "x_ll"]
    for size in feature_size_list:
        sub_list = [col for col in column_list if col.__contains__(size)]
        print(size, len(sub_list))


print("Feature_Count:", len(x_columns))
feature_size_count(x_columns)

for col in x_columns:
    if ("x_xs" not in col) & ("x_ss" not in col) &("x_mm" not in col)&("x_ll" not in col):
        print("\nMislabeled Column:", col)

print("\nTarget_Values:", y_columns)

Feature_Count: 483
x_xs 46
x_ss 53
x_mm 143
x_ll 241

Target_Values: ['y_album_buying', 'y_heal_b_kpop']


#### Set Feature List

In [60]:
healing_feature_list = ["x_xs_demo_heal_group_num"]
album_buying_feature_list = ["x_xs_beha_alb_typ_group_num", "x_xs_beha_alb_cnt_group_num"] 

# top_a_features = [
#     "x_xs_beha_evt_live",
#     "x_xs_demo_regi_group_num",
#     #"x_xs_demo_heal_group_num",
#     "x_xs_beha_mf_group_num",
#     "x_xs_artm_mpp_memb_n",
#     "x_xs_comp_dis_sent",
#     "x_xs_demo_gen_group_num",
#     "x_xs_comp_west_sent",
#     "x_ss_demo_role_prof_e",
#     "x_ss_artm_mpp_memb_n_bit_0",
#     "x_ss_demo_vert_middle",
#     "x_ss_demo_vert_intro",
#     "x_ss_demo_vert_depends",
#     "x_ss_demo_vert_extro",
#     "x_ss_demo_role_stud_u",
#     "x_ss_demo_role_prof_u",
#     "x_ss_demo_role_stud_e",
#     "x_ss_demo_sex_other",
#     #"x_mm_demo_regi_america_n", #Correlated
#     "x_mm_altm_jglt_dy6",
#     "x_ll_memb_soe_Dahyun_mnh",
#     "x_ll_memb_soe_Jihyo_btd",
#     "x_ll_memb_soe_Jihyo_loa",
#     "x_ll_memb_soe_Sana_loa",
#     "x_ll_memb_soe_Tzuyu_none",
#     "x_ll_memb_soe_Jihyo_pds",
#     "x_ll_memb_soe_Nayeon_sgl",
#     # Extra Features
#     # "x_xs_beha_evt_tour", # Repeate # Correlated
#     "x_mm_demo_regi_america_c_s",  # Repeate
#     "x_ll_memb_soe_Dahyun_knk", # Repeate
#     "x_ll_memb_soe_Chaeyoung_csm",

#     # New Extra Features
#     "x_xs_comp_west_sent_japn", #

#     ## Extra B Features
#     # "x_xs_beha_evt_kcon",
#     # "x_ll_memb_soe_Sana_yoy",
#     # "x_mm_demo_regi_other_other",
#     # "x_mm_demo_ag_000_014",
#     # "x_ll_memb_soe_Dahyun_loa",


#     ## Extra B Features for Healing
#     # "x_xs_beha_browse_pm", # : 3
#     # "x_mm_memb_cr_chaeyoung", #: 2
#     # "x_ss_artm_mpp_memb_n_bit_3", ##, : 2
#     # "x_ll_altm_omglt_allofthemnwhateverisgood", # : 2
#     # "x_xs_artm_ovrl_cbk_n", #: 2
#     # "x_mm_memb_fvt_shp_mina", #: 2

#     ## PCA Features

#     #### Alternative Music PCAs
#     "x_xs_altm_pca_a1",

#     #"x_xs_altm_pca_b1",
#     #"x_xs_altm_pca_b2",

#     #### Member PCAs
#     "x_xs_memb_pca_a1",
    
#     #"x_xs_memb_pca_b1",
#     #"x_xs_memb_pca_b2",
#     #"x_xs_memb_pca_b3",

#     #### Artistic Medium PCAs
#     "x_xs_artm_pca_a1",


#     ## Classification Feature
#     #"x_xs_km_clstr",
#     ]

top_a_features = [
    # "x_xs_beha_evt_live",
    "x_xs_beha_evt_live_tour", # Not In final Version 
    "x_xs_demo_regi_group_num",
    #"x_xs_demo_heal_group_num",
    "x_xs_beha_mf_group_num",
    "x_xs_artm_mpp_memb_n",
    "x_xs_comp_dis_sent",
    "x_xs_demo_gen_group_num",
    "x_xs_comp_west_sent",
    "x_ss_artm_mpp_memb_n_bit_0",
    "x_xs_demo_vert_group_num", # Not In final Version 
    #"x_ss_demo_vert_middle", # Reduced
    #"x_ss_demo_vert_intro", # Reduced
    #"x_ss_demo_vert_depends", # Reduced
    #"x_ss_demo_vert_extro", # Reduced
    "x_xs_demo_role_student", # Not In final Version 
    "x_xs_demo_role_employed",
    #"x_ss_demo_role_stud_u",# Reduced
    #"x_ss_demo_role_prof_u",# Reduced
    #"x_ss_demo_role_stud_e",# Reduced
    #"x_ss_demo_role_prof_e",# Reduced
    "x_ss_demo_sex_other",
    #"x_mm_demo_regi_america_n", ,# Reduced
    "x_mm_altm_jglt_dy6",
    "x_ll_memb_soe_Dahyun_mnh",
    "x_ll_memb_soe_Jihyo_btd",
    "x_ll_memb_soe_Jihyo_loa",
    "x_ll_memb_soe_Sana_loa",
    "x_ll_memb_soe_Tzuyu_none",
    "x_ll_memb_soe_Jihyo_pds",
    "x_ll_memb_soe_Nayeon_sgl",
    # Extra Features
    #"x_xs_beha_evt_tour", # Repeate ,# Reduced
    "x_mm_demo_regi_america_c_s",  # Repeate
    "x_ll_memb_soe_Dahyun_knk", # Repeate
    "x_ll_memb_soe_Chaeyoung_csm",

    # New Extra Features
    "x_xs_comp_west_sent_japn", #

    ## Extra B Features
    # "x_xs_beha_evt_kcon",
    # "x_ll_memb_soe_Sana_yoy",
    # "x_mm_demo_regi_other_other",
    # "x_mm_demo_ag_000_014",
    # "x_ll_memb_soe_Dahyun_loa",


    ## Extra B Features for Healing
    # "x_xs_beha_browse_pm", # : 3
    # "x_mm_memb_cr_chaeyoung", #: 2
    # "x_ss_artm_mpp_memb_n_bit_3", ##, : 2
    # "x_ll_altm_omglt_allofthemnwhateverisgood", # : 2
    # "x_xs_artm_ovrl_cbk_n", #: 2
    # "x_mm_memb_fvt_shp_mina", #: 2

    ## PCA Features

    #### Alternative Music PCAs
    "x_xs_altm_pca_a1",

    #"x_xs_altm_pca_b1",
    #"x_xs_altm_pca_b2",

    #### Member PCAs
    "x_xs_memb_pca_a1",
    
    #"x_xs_memb_pca_b1",
    #"x_xs_memb_pca_b2",
    #"x_xs_memb_pca_b3",

    #### Artistic Medium PCAs
    "x_xs_artm_pca_a1",


    ## Classification Feature
    #"x_xs_km_clstr",
    ]





if target == "y_album_buying": # y_heal_b_kpop
    top_a_features = top_a_features + healing_feature_list

if target == "y_heal_b_kpop":
    top_a_features = top_a_features + album_buying_feature_list



top_b_features = [
    "x_xs_artm_fmv_japn",
    "x_xs_artm_ftbt_japn",
    "x_xs_artm_ovrl_cbk_n",
    "x_xs_demo_ag_num",
    "x_xs_beha_browse_pm",
    "x_xs_comp_rrp_sent",
    "x_xs_comp_lntv_sent",
    "x_xs_beha_evt_kcon", # Above
    "x_xs_beha_evt_tour", # Above
    "x_ss_artm_fts21_n_bit_2",
    "x_ss_artm_fcho_n_bit_3",
    "x_ss_artm_mpp_memb_n_bit_1",
    "x_ss_artm_ftts_n_bit_3",
    "x_ss_artm_fmv_n_bit_4",
    "x_ss_artm_mpp_memb_n_bit_3",
    "x_ss_artm_ftts_n_bit_2",
    "x_ss_artm_ovrl_cbk_n_bit_3",
    "x_mm_beha_intro_blackpink",
    "x_mm_beha_rfc_tty",
    "x_mm_artm_lfts_n_bit_2",
    "x_mm_altm_jglt_sks",
    "x_mm_memb_fvt_shp_mina",
    "x_mm_artm_lfts_n_bit_5",
    "x_mm_artm_ftbt_n_bit_2",
    "x_mm_beha_ett_yb",
    "x_mm_memb_cr_momo",
    "x_mm_memb_cr_mina",
    "x_mm_memb_cr_nayeon",
    "x_mm_memb_cr_chaeyoung",
    "x_mm_memb_cr_tzuyu",
    "x_mm_memb_cr_jeongyeon",
    "x_mm_memb_cr_jihyo",
    "x_mm_memb_cr_sana",
    "x_mm_demo_ag_000_014", #Above
    "x_mm_demo_regi_america_c_s", #Above
    "x_mm_beha_rfc_vlu",
    "x_mm_demo_regi_other_other",
    "x_mm_beha_intro_idk",
    "x_mm_memb_bias_nayeon",
    "x_mm_memb_soy_2021_jeongyeon",
    "x_mm_memb_bias_wkr_jeongyeon",
    "x_mm_memb_bias_wkr_dahyun",
    "x_ll_memb_soe_Momo_btd",
    "x_ll_memb_soe_Sana_none",
    "x_ll_memb_soe_Sana_yoy",
    "x_ll_altm_oglt_bigbang",
    "x_ll_memb_soe_Chaeyoung_nan",
    "x_ll_altm_omglt_westernpop",
    "x_ll_altm_oglt_snsd",
    "x_ll_altm_oglt_loona",
    "x_ll_altm_omglt_alternative",
    "x_ll_altm_omglt_allofthemnwhateverisgood",
    "x_ll_altm_oglt_seventeen",
    "x_ll_altm_oglt_iz*one",
    "x_ll_altm_oglt_bts",
    #"x_ll_memb_soe_Dahyun_knk",
    "x_ll_memb_soe_Dahyun_loa",
    "x_ll_memb_soe_Mina_knk"
]


bottom_features = [
"x_xs_misc_pop_group_nu",
"x_xs_artm_mpp_year",
"x_xs_artm_ftts_japn",
"x_xs_comp_cont_sent",
"x_xs_beha_evt_show",
"x_xs_comp_jpn_sent",
"x_xs_demo_role_student",
"x_ss_artm_song_pref3_s",
"x_ss_artm_ovrl_cbk_n_bit_0",
"x_ss_artm_fts21_n_bit_3",
"x_ss_artm_fcho_n_bit_3",
"x_ss_artm_fcho_n_bit_4",
"x_ss_artm_fmv_n_bit_2",
"x_ss_artm_fmv_n_bit_0",
"x_ss_artm_ovrl_cbk_n_bit_3",
"x_ss_demo_sex_gay_lesbian",
"x_mm_beha_rfc_vlu",
"x_mm_artm_ftbt_n_bit_6",
"x_ll_altm_oglt_mamamoo",
"x_ll_altm_oglt_itzy",
"x_ll_altm_oglt_dreamcatcher",
"x_ll_altm_omglt_rapnhiphop",
"x_ll_altm_oglt_snsd",
# Extra Features
"x_ll_memb_soe_Dahyun_none"
]

#### Filter Down Features

In [61]:
only_top_features = True #Main
only_top_b_features = False
only_middle_features = False
no_bottom_features = False
subgroup = "x_ll" # False  x_xs x_ss  x_mm x_ll
subgrouping = False


if only_top_features:
    print("Using only top features")#
    x_columns = top_a_features #

if only_top_b_features:
    print("Using only top b features")#
    x_columns = top_b_features #

if only_middle_features:
    print("Using only middle features")#
    x_columns = [col for col in x_columns if col not in top_a_features + bottom_features]

if no_bottom_features:
    print("Using no bottom features")#
    x_columns = [col for col in x_columns if col not in bottom_features]

if subgrouping:
    print("Using subset of features")#
    x_columns = [col for col in df_clean.columns if col.startswith(subgroup)] #


print("Features Used:", len(x_columns))

Using only top features
Features Used: 28


In [62]:
df_clean = df_clean[["split"] + [target] + x_columns]
df_clean.head()

,split,y_album_buying,x_xs_beha_evt_live_tour,x_xs_demo_regi_group_num,x_xs_beha_mf_group_num,x_xs_artm_mpp_memb_n,x_xs_comp_dis_sent,x_xs_demo_gen_group_num,x_xs_comp_west_sent,x_ss_artm_mpp_memb_n_bit_0,...,x_ll_memb_soe_Jihyo_pds,x_ll_memb_soe_Nayeon_sgl,x_mm_demo_regi_america_c_s,x_ll_memb_soe_Dahyun_knk,x_ll_memb_soe_Chaeyoung_csm,x_xs_comp_west_sent_japn,x_xs_altm_pca_a1,x_xs_memb_pca_a1,x_xs_artm_pca_a1,x_xs_demo_heal_group_num
0,test,1.0,1,6.0,60.0,6.0,5.0,0.0,4.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.528840,-0.027170,1.005668,2.0
1,test,1.0,2,6.0,6.0,6.0,5.0,1.0,3.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,-0.422772,-0.803847,0.181624,2.0
2,test,0.0,0,1.0,12.0,4.0,5.0,1.0,2.0,0.0,...,1.0,0.0,0.0,0.0,1.0,0.0,1.021795,-0.396969,-1.044984,0.0
3,train,0.0,0,6.0,1.0,0.0,4.0,1.0,5.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,-0.675254,0.204281,-0.870150,2.0
4,train,1.0,6,6.0,84.0,6.0,5.0,1.0,3.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.686089,1.418075,0.160196,2.0


## Train Test Split

In [63]:
X_train = df_clean[df_clean["split"]=="train"][x_columns]
X_test = df_clean[df_clean["split"]=="test"][x_columns]
y_train = df_clean[df_clean["split"]=="train"][[target]]
y_test = df_clean[df_clean["split"]=="test"][[target]]


# X_train, X_test, y_train, y_test = train_test_split(
#     df_clean[x_columns],
#     df_clean[y_columns],
#     random_state=42,
#     test_size=0.3
# )

## Scale Features

In [64]:
scaler = MinMaxScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

X_train = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)

## Generate Score Dictionary

In [65]:
score_dict = {}

## Generate Baseline

#### Generate Baselines

In [66]:
target_tag_01 = y_train[target].value_counts()[1.0]
target_total = len(y_train)
target_rate = round(target_tag_01/target_total, 2)
print(target_tag_01, target_total)
print(target_rate)

341 642
0.53


In [67]:
target_tag_01 = y_test[target].value_counts()[1.0]
target_total = len(y_test)
target_rate = round(target_tag_01/target_total, 2)
print(target_tag_01, target_total)
print(target_rate)

130 263
0.49


In [68]:
def aprf_funct(probability, y_test):
    cnt = len(y_test) 
    tp = y_test[target].value_counts()[1.0]
    tn = cnt - tp
    base_score_dict = {}
    base_score_dict["accuracy"] = ((tp * probability) + (tn * (1 - probability))) / cnt
    base_score_dict["precision"] = (tp * probability) / ((tp * probability) + (tn * probability)) if (tp * probability) + (tn * probability) != 0 else 0
    base_score_dict["recall"] = (tp * probability) / tp if tp != 0 else 0
    base_score_dict["f1-score"] = (2 * base_score_dict["precision"] * base_score_dict["recall"]) / (base_score_dict["precision"] + base_score_dict["recall"]) if (base_score_dict["precision"] + base_score_dict["recall"]) != 0 else 0
    return base_score_dict

print(aprf_funct(1.0, y_test))

{'accuracy': 0.49429657794676807, 'precision': 0.49429657794676807, 'recall': 1.0, 'f1-score': 0.6615776081424937}


In [69]:
target_tag_01 = y_train[target].value_counts()[1.0]
target_total = len(y_train)
target_rate = round(target_tag_01/target_total, 2)

baseline_dict = {
    "baseline_rand": 0.5, # Guessing 50/50
    "baseline_crt": target_rate, # Guessing based on rate in Training Set
    "baseline_100": 1.00, # Assume they are all 1
}

for key, value in baseline_dict.items():
    baseline_score_dict = aprf_funct(value, y_test)
    score_dict[key] = {
        "precision":round(baseline_score_dict["precision"], 2),
        "recall":round(baseline_score_dict["recall"], 2),
        "f1-score":round(baseline_score_dict["f1-score"], 2),
        "accuracy":round(baseline_score_dict["accuracy"], 2)
        }
score_dict

{'baseline_rand': {'precision': 0.49,
  'recall': 0.5,
  'f1-score': 0.5,
  'accuracy': 0.5},
 'baseline_crt': {'precision': 0.49,
  'recall': 0.53,
  'f1-score': 0.51,
  'accuracy': 0.5},
 'baseline_100': {'precision': 0.49,
  'recall': 1.0,
  'f1-score': 0.66,
  'accuracy': 0.49}}

In [70]:
# import matplotlib.pyplot as plt
# import numpy as np

# # Define the range of probability values
# probabilities = np.linspace(0, 1, 100)  # 100 points between 0 and 1

# # Calculate accuracy and F1 score for each probability
# accuracies = []
# f1_scores = []
# for prob in probabilities:
#     accuracy, f1s = aprf_funct(prob)
#     accuracies.append(accuracy)
#     f1_scores.append(f1s)

# # Plot the results
# plt.figure(figsize=(10, 6))
# plt.plot(probabilities, accuracies, label='Accuracy', color='blue')
# plt.plot(probabilities, f1_scores, label='F1 Score', color='red')
# plt.xlabel('Probability')
# plt.ylabel('Value')
# plt.title('Accuracy and F1 Score vs. Probability')
# plt.legend()
# plt.grid(True)
# plt.show()

## Regression Modeling

#### Basic Logistic Regression

In [71]:
model = LogisticRegression(
    random_state=0,
    penalty=None, #l1 #l2
    #solver="liblinear"
    ).fit(X_train, y_train[target])

log_model = model
y_pred = model.predict(X_test)

cr = classification_report(y_test[target], y_pred, output_dict=True)
score_dict["model_logistic"] = {
    "precision":round(cr["1.0"]["precision"], 2),
    "recall":round(cr["1.0"]["recall"], 2),
    "f1-score":round(cr["1.0"]["f1-score"], 2),
    "accuracy":round(cr["accuracy"], 2)
}
score_dict

{'baseline_rand': {'precision': 0.49,
  'recall': 0.5,
  'f1-score': 0.5,
  'accuracy': 0.5},
 'baseline_crt': {'precision': 0.49,
  'recall': 0.53,
  'f1-score': 0.51,
  'accuracy': 0.5},
 'baseline_100': {'precision': 0.49,
  'recall': 1.0,
  'f1-score': 0.66,
  'accuracy': 0.49},
 'model_logistic': {'precision': 0.63,
  'recall': 0.77,
  'f1-score': 0.69,
  'accuracy': 0.67}}

In [72]:
model_logistic_coef_df = get_coefficient_df(model, x_columns)
print(model_logistic_coef_df.head(15))  # Show top 10 features by importance

                        Feature  Coefficient
13     x_ll_memb_soe_Dahyun_mnh     6.334601
14      x_ll_memb_soe_Jihyo_btd    -3.248133
0       x_xs_beha_evt_live_tour     2.749243
19     x_ll_memb_soe_Nayeon_sgl     2.617840
16       x_ll_memb_soe_Sana_loa    -2.419921
1      x_xs_demo_regi_group_num     1.500606
6           x_xs_comp_west_sent     1.198926
18      x_ll_memb_soe_Jihyo_pds    -1.138560
21     x_ll_memb_soe_Dahyun_knk     1.075072
23     x_xs_comp_west_sent_japn     1.062102
5       x_xs_demo_gen_group_num    -1.061062
15      x_ll_memb_soe_Jihyo_loa     1.052096
27     x_xs_demo_heal_group_num     1.041526
22  x_ll_memb_soe_Chaeyoung_csm     0.942300
2        x_xs_beha_mf_group_num     0.914174


In [73]:
print(model_logistic_coef_df.tail(15))  # Show top 10 features by importance

                        Feature  Coefficient
22  x_ll_memb_soe_Chaeyoung_csm     0.942300
2        x_xs_beha_mf_group_num     0.914174
20   x_mm_demo_regi_america_c_s    -0.905180
17     x_ll_memb_soe_Tzuyu_none    -0.896302
3          x_xs_artm_mpp_memb_n     0.883426
4            x_xs_comp_dis_sent     0.782825
24             x_xs_altm_pca_a1     0.684569
25             x_xs_memb_pca_a1    -0.670180
26             x_xs_artm_pca_a1    -0.560508
11          x_ss_demo_sex_other    -0.422518
12           x_mm_altm_jglt_dy6     0.325693
10      x_xs_demo_role_employed     0.130153
9        x_xs_demo_role_student    -0.119922
8      x_xs_demo_vert_group_num    -0.090793
7    x_ss_artm_mpp_memb_n_bit_0    -0.028771


#### Tuned Logistic Regression

In [74]:
model = LogisticRegression(penalty='l1', solver='liblinear') #l1

# Hyperparameter grid to search over
param_grid = {
   'C': [0.001, 0.01, 0.1, 1.0, 10.0, 100.0],  # Regularization strength (inverse of regularization parameter)
   'max_iter': [1000, 2000, 3000],  # Maximum number of iterations
   'tol': [1e-4, 1e-3, 1e-2],  # Tolerance for stopping criteria
   'class_weight': [None, 'balanced']  # Class weighting strategy
}

# Initialize GridSearchCV with cross-validation
grid_search = GridSearchCV(estimator=model, param_grid=param_grid, cv=5, scoring='accuracy', n_jobs=-1) #f1

# Fit the grid search to the training data
grid_search.fit(X_train, y_train[target])

# Use the best model from the grid search to make predictions on the test set
best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test)

cr = classification_report(y_test[target], y_pred, output_dict=True)
score_dict["logistic_l1"] = {
   "precision": round(cr["1.0"]["precision"], 2),
   "recall": round(cr["1.0"]["recall"], 2),
   "f1-score": round(cr["1.0"]["f1-score"], 2),
   "accuracy": round(cr["accuracy"], 2),
   "n_features_used": sum(best_model.coef_[0] != 0)  # Count non-zero coefficients (features selected)
}
score_dict

{'baseline_rand': {'precision': 0.49,
  'recall': 0.5,
  'f1-score': 0.5,
  'accuracy': 0.5},
 'baseline_crt': {'precision': 0.49,
  'recall': 0.53,
  'f1-score': 0.51,
  'accuracy': 0.5},
 'baseline_100': {'precision': 0.49,
  'recall': 1.0,
  'f1-score': 0.66,
  'accuracy': 0.49},
 'model_logistic': {'precision': 0.63,
  'recall': 0.77,
  'f1-score': 0.69,
  'accuracy': 0.67},
 'logistic_l1': {'precision': 0.68,
  'recall': 0.75,
  'f1-score': 0.71,
  'accuracy': 0.7,
  'n_features_used': 28}}

In [75]:

# Example usage:
logistic_l1_coef_df = get_coefficient_df(best_model, x_columns)
print(logistic_l1_coef_df.head(15))  # Show top 10 features by importance

                        Feature  Coefficient
13     x_ll_memb_soe_Dahyun_mnh     3.731586
14      x_ll_memb_soe_Jihyo_btd    -2.902154
0       x_xs_beha_evt_live_tour     2.672402
19     x_ll_memb_soe_Nayeon_sgl     2.578113
16       x_ll_memb_soe_Sana_loa    -2.175750
1      x_xs_demo_regi_group_num     1.430916
6           x_xs_comp_west_sent     1.090148
5       x_xs_demo_gen_group_num    -1.058102
21     x_ll_memb_soe_Dahyun_knk     1.033575
27     x_xs_demo_heal_group_num     1.018915
18      x_ll_memb_soe_Jihyo_pds    -0.971275
22  x_ll_memb_soe_Chaeyoung_csm     0.923077
15      x_ll_memb_soe_Jihyo_loa     0.895225
23     x_xs_comp_west_sent_japn     0.894111
17     x_ll_memb_soe_Tzuyu_none    -0.875095


In [110]:
len(logistic_l1_coef_df)

28

In [116]:
logistic_l1_coef_df.to_excel("outputData/model_l1_coef.xlsx")

In [76]:
print(logistic_l1_coef_df.tail(15))  # Show top 10 features by importance

                       Feature  Coefficient
23    x_xs_comp_west_sent_japn     0.894111
17    x_ll_memb_soe_Tzuyu_none    -0.875095
20  x_mm_demo_regi_america_c_s    -0.873646
2       x_xs_beha_mf_group_num     0.868880
3         x_xs_artm_mpp_memb_n     0.853028
4           x_xs_comp_dis_sent     0.740695
25            x_xs_memb_pca_a1    -0.667133
24            x_xs_altm_pca_a1     0.644443
26            x_xs_artm_pca_a1    -0.576162
11         x_ss_demo_sex_other    -0.394513
12          x_mm_altm_jglt_dy6     0.322591
9       x_xs_demo_role_student    -0.127354
8     x_xs_demo_vert_group_num    -0.125065
10     x_xs_demo_role_employed     0.113901
7   x_ss_artm_mpp_memb_n_bit_0    -0.022582


In [77]:
#X_train["x_xs_demo_ag_num"].value_counts()

In [78]:
# Baseline: 0.6286764705882353
# New: 0.6537

## Decision Tree Models

In [79]:
from sklearn.tree import DecisionTreeClassifier

model = DecisionTreeClassifier(max_depth=2, random_state=0)
model.fit(X_train, y_train[target])
y_pred = model.predict(X_test)

In [80]:
cr = classification_report(y_test[target], y_pred, output_dict=True)
score_dict["decision_tree"] = {
    "precision":round(cr["1.0"]["precision"], 2),
    "recall":round(cr["1.0"]["recall"], 2),
    "f1-score":round(cr["1.0"]["f1-score"], 2),
    "accuracy":round(cr["accuracy"], 2)
}
score_dict

{'baseline_rand': {'precision': 0.49,
  'recall': 0.5,
  'f1-score': 0.5,
  'accuracy': 0.5},
 'baseline_crt': {'precision': 0.49,
  'recall': 0.53,
  'f1-score': 0.51,
  'accuracy': 0.5},
 'baseline_100': {'precision': 0.49,
  'recall': 1.0,
  'f1-score': 0.66,
  'accuracy': 0.49},
 'model_logistic': {'precision': 0.63,
  'recall': 0.77,
  'f1-score': 0.69,
  'accuracy': 0.67},
 'logistic_l1': {'precision': 0.68,
  'recall': 0.75,
  'f1-score': 0.71,
  'accuracy': 0.7,
  'n_features_used': 28},
 'decision_tree': {'precision': 0.62,
  'recall': 0.79,
  'f1-score': 0.7,
  'accuracy': 0.66}}

{'baseline_50': {'precision': 0.57,
  'recall': 0.58,
  'f1-score': 0.58,
  'accuracy': 0.56},
 'baseline_52': {'precision': 0.56,
  'recall': 0.6,
  'f1-score': 0.58,
  'accuracy': 0.54},
 'model_logistic': {'precision': 0.62,
  'recall': 0.64,
  'f1-score': 0.63,
  'accuracy': 0.61},
 'logistic_l1': {'precision': 0.6,
  'recall': 0.62,
  'f1-score': 0.61,
  'accuracy': 0.59,
  'n_features_used': 88},
 'decision_tree': {'precision': 0.58,
  'recall': 0.62,
  'f1-score': 0.6,
  'accuracy': 0.57}}

#### Random Forest

In [81]:
from sklearn.ensemble import RandomForestClassifier

# Initialize Random Forest model
model = RandomForestClassifier(random_state=42)

# Hyperparameter grid to search over
param_grid = {
   'n_estimators': [125], #[100, 200, 300],  # Number of trees (100) (125)
   'max_depth': [7], #[None, 10, 20, 30],  # Maximum depth of trees (10) (7)
   'min_samples_split': [5], #[2, 5, 10],  # Minimum samples required to split (5)
   'min_samples_leaf': [2],  #[1, 2, 4],  # Minimum samples required at leaf node (2)
   'max_features': ['sqrt'], #['sqrt', 'log2', None]  # Number of features to consider for best split (sqrt)
}

# Initialize GridSearchCV with cross-validation
grid_search = GridSearchCV(estimator=model, param_grid=param_grid, cv=5, scoring='accuracy', n_jobs=-1)

# Fit the grid search to the training data
grid_search.fit(X_train, y_train[target])

# Use the best model from the grid search to make predictions on the test set
best_model = grid_search.best_estimator_
rf_model = best_model
y_pred = best_model.predict(X_test)

cr = classification_report(y_test[target], y_pred, output_dict=True)
score_dict["random_forest"] = {
   "precision": round(cr["1.0"]["precision"], 2),
   "recall": round(cr["1.0"]["recall"], 2),
   "f1-score": round(cr["1.0"]["f1-score"], 2),
   "accuracy": round(cr["accuracy"], 2),
#    "feature_importance": dict(sorted(zip(X_train.columns, 
#                                          best_model.feature_importances_),
#                                      key=lambda x: x[1], 
#                                      reverse=True)[:10])  # Top 10 features
}
score_dict["random_forest"]

{'precision': 0.67, 'recall': 0.82, 'f1-score': 0.73, 'accuracy': 0.71}

In [82]:
{'precision': 0.64, 'recall': 0.75, 'f1-score': 0.69, 'accuracy': 0.67}

{'precision': 0.64, 'recall': 0.75, 'f1-score': 0.69, 'accuracy': 0.67}

In [83]:
print("n_estimators", rf_model.get_params()["n_estimators"])
print("max_depth", rf_model.get_params()["max_depth"])
print("min_samples_split", rf_model.get_params()["min_samples_split"])

print("min_samples_leaf", rf_model.get_params()["min_samples_leaf"])

n_estimators 125
max_depth 7
min_samples_split 5
min_samples_leaf 2


In [84]:
print(rf_model.get_params()["n_estimators"])

125


In [85]:
print(rf_model.get_params()["n_estimators"])

125


In [86]:
rf_t20 = dict(sorted(zip(X_train.columns, 
                best_model.feature_importances_),
            key=lambda x: abs(x[1]), 
            reverse=True)[:20])

rf_t20

{'x_xs_memb_pca_a1': 0.1196928309056993,
 'x_xs_artm_pca_a1': 0.11206608709908221,
 'x_xs_altm_pca_a1': 0.10770621717027258,
 'x_xs_beha_mf_group_num': 0.08545956243955463,
 'x_xs_demo_regi_group_num': 0.0834850649750794,
 'x_xs_demo_heal_group_num': 0.08342929810500789,
 'x_xs_beha_evt_live_tour': 0.07546899378138212,
 'x_xs_artm_mpp_memb_n': 0.06913049578464558,
 'x_xs_comp_dis_sent': 0.04573114322797571,
 'x_xs_comp_west_sent': 0.04547038816933292,
 'x_xs_demo_vert_group_num': 0.03242141427459094,
 'x_mm_altm_jglt_dy6': 0.024446836835798118,
 'x_xs_demo_gen_group_num': 0.020781602784908596,
 'x_xs_demo_role_employed': 0.02006059067148361,
 'x_xs_demo_role_student': 0.018221365791083605,
 'x_ll_memb_soe_Chaeyoung_csm': 0.012961007628621578,
 'x_mm_demo_regi_america_c_s': 0.008899018980738919,
 'x_ss_artm_mpp_memb_n_bit_0': 0.008339395870609764,
 'x_ll_memb_soe_Tzuyu_none': 0.0068336660740168625,
 'x_ll_memb_soe_Dahyun_knk': 0.0048800586189121494}

In [87]:
rf_b20 = dict(sorted(zip(X_train.columns, 
                best_model.feature_importances_),
            key=lambda x: abs(x[1]), 
            reverse=True)[-21:])

rf_b20

{'x_xs_artm_mpp_memb_n': 0.06913049578464558,
 'x_xs_comp_dis_sent': 0.04573114322797571,
 'x_xs_comp_west_sent': 0.04547038816933292,
 'x_xs_demo_vert_group_num': 0.03242141427459094,
 'x_mm_altm_jglt_dy6': 0.024446836835798118,
 'x_xs_demo_gen_group_num': 0.020781602784908596,
 'x_xs_demo_role_employed': 0.02006059067148361,
 'x_xs_demo_role_student': 0.018221365791083605,
 'x_ll_memb_soe_Chaeyoung_csm': 0.012961007628621578,
 'x_mm_demo_regi_america_c_s': 0.008899018980738919,
 'x_ss_artm_mpp_memb_n_bit_0': 0.008339395870609764,
 'x_ll_memb_soe_Tzuyu_none': 0.0068336660740168625,
 'x_ll_memb_soe_Dahyun_knk': 0.0048800586189121494,
 'x_ll_memb_soe_Jihyo_loa': 0.004059940610336384,
 'x_ss_demo_sex_other': 0.002941405686577154,
 'x_ll_memb_soe_Jihyo_btd': 0.002406484680702629,
 'x_ll_memb_soe_Nayeon_sgl': 0.0020926203963397344,
 'x_ll_memb_soe_Jihyo_pds': 0.001113135593231208,
 'x_xs_comp_west_sent_japn': 0.0009404482807570144,
 'x_ll_memb_soe_Sana_loa': 0.0008370784865667994,
 'x_ll_m

In [88]:
# from sklearn.tree import DecisionTreeClassifier
# from sklearn.model_selection import GridSearchCV
# from sklearn.metrics import roc_auc_score, f1_score, accuracy_score

# # Define parameter grid
# param_grid = {
#     'max_depth': [2, 3, 4, 5, 6, 8, 10, None],
#     'min_samples_split': [2, 5, 10, 20],
#     'min_samples_leaf': [1, 2, 4, 8],
#     'criterion': ['gini', 'entropy'],
#     'splitter': ['best', 'random'],
#     'class_weight': [None, 'balanced']
# }

# # Create base model
# base_model = DecisionTreeClassifier(random_state=0)

# # Set up GridSearchCV
# grid_search = GridSearchCV(
#     estimator=base_model,
#     param_grid=param_grid,
#     scoring='roc_auc',  # Can also use 'f1', 'accuracy', etc.
#     cv=5,              # 5-fold cross-validation
#     verbose=1,
#     n_jobs=-1          # Use all available cores
# )

# # Fit the grid search
# grid_search.fit(X_train, y_train)

# # Best parameters and scores
# print(f"Best parameters: {grid_search.best_params_}")
# print(f"Best cross-validation score: {grid_search.best_score_:.4f}")

# # Get the best model
# best_model = grid_search.best_estimator_

# # Evaluate on test set
# y_pred = best_model.predict(X_test)
# y_pred_proba = best_model.predict_proba(X_test)[:,1]

# test_accuracy = accuracy_score(y_test, y_pred)
# test_f1 = f1_score(y_test, y_pred)
# test_auc = roc_auc_score(y_test, y_pred_proba)

# print(f"Test Accuracy: {test_accuracy:.4f}")
# print(f"Test F1 Score: {test_f1:.4f}")
# print(f"Test AUC: {test_auc:.4f}")

#### Gradient Boosting Classifier

In [89]:
# from sklearn.ensemble import GradientBoostingClassifier
# from sklearn.model_selection import GridSearchCV
# from sklearn.metrics import accuracy_score, roc_auc_score, f1_score

# # Define parameter grid
# param_grid = {
#     'n_estimators': [50, 100, 200, 300],
#     'learning_rate': [0.01, 0.05, 0.1, 0.2],
#     'max_depth': [3, 4, 5, 6],
#     'min_samples_split': [2, 5, 10],
#     'min_samples_leaf': [1, 2, 4],
#     'subsample': [0.8, 0.9, 1.0],
#     'max_features': ['sqrt', 'log2', None]
# }

# # Create base model
# base_model = GradientBoostingClassifier(random_state=0)

# # Set up scoring metrics
# scoring = {
#     'AUC': make_scorer(roc_auc_score, needs_proba=True),
#     'f1': make_scorer(f1_score),
#     'accuracy': make_scorer(accuracy_score)
# }

# # Set up GridSearchCV
# grid_search = GridSearchCV(
#     estimator=base_model,
#     param_grid=param_grid,
#     scoring=scoring,
#     refit='f1',      # Optimize for AUC
#     cv=5,             # 5-fold cross-validation
#     verbose=1,
#     n_jobs=-1         # Use all available cores
# )

# # Fit the grid search
# grid_search.fit(X_train, y_train)

# # Best parameters and scores
# print(f"Best parameters: {grid_search.best_params_}")
# print(f"Best cross-validation AUC: {grid_search.best_score_:.4f}")

# # Get the best model
# best_model = grid_search.best_estimator_
# gbdt_model = best_model

# # Evaluate on test set
# y_pred = best_model.predict(X_test)
# y_pred_proba = best_model.predict_proba(X_test)[:,1]

# test_accuracy = accuracy_score(y_test, y_pred)
# test_f1 = f1_score(y_test, y_pred)
# test_auc = roc_auc_score(y_test, y_pred_proba)

# cr = classification_report(y_test["y_album_buying"], y_pred, output_dict=True)
# score_dict["gbdt"] = {
#    "precision": round(cr["1.0"]["precision"], 2),
#    "recall": round(cr["1.0"]["recall"], 2),
#    "f1-score": round(cr["1.0"]["f1-score"], 2),
#    "accuracy": round(cr["accuracy"], 2),
# #    "feature_importance": dict(sorted(zip(X_train.columns, 
# #                                          best_model.feature_importances_),
# #                                      key=lambda x: x[1], 
# #                                      reverse=True)[:10])  # Top 10 features
# }
# score_dict["gbdt"]

# ## Feature importance of best model
# #feature_importance = best_model.feature_importances_

In [90]:
#gbdt_model.get_params()

## Suppot Vector Machines

In [91]:
# from sklearn.svm import SVC

# # Initialize SVM model with RBF kernel
# model = SVC(kernel='rbf')

# # Hyperparameter grid to search over
# param_grid = {
#    'C': [0.1, 1.0, 10.0, 100.0],  # Regularization parameter
#    'gamma': ['scale', 'auto', 0.001, 0.01, 0.1, 1],  # Kernel coefficient
#    'tol': [1e-4, 1e-3],  # Tolerance for stopping criteria
#    'class_weight': [None, 'balanced']  # Class weighting strategy
# }

# # Initialize GridSearchCV with cross-validation
# grid_search = GridSearchCV(estimator=model, param_grid=param_grid, cv=5, scoring='accuracy', n_jobs=-1)

# # Fit the grid search to the training data
# grid_search.fit(X_train, y_train["y_album_buying"])

# # Use the best model from the grid search to make predictions on the test set
# best_model = grid_search.best_estimator_
# y_pred = best_model.predict(X_test)

# cr = classification_report(y_test["y_album_buying"], y_pred, output_dict=True)
# score_dict["svm_rbf"] = {
#    "precision": round(cr["1.0"]["precision"], 2),
#    "recall": round(cr["1.0"]["recall"], 2),
#    "f1-score": round(cr["1.0"]["f1-score"], 2),
#    "accuracy": round(cr["accuracy"], 2),
#    #"best_params": grid_search.best_params_
# }
# score_dict["svm_rbf"] 

#### Linear SVC

In [92]:
model = LinearSVC(random_state=0)
# Hyperparameter grid to search over
param_grid = {
    'C': [.001, 0.01, 0.1, 1.0, 10.0, 100.0, 1000.0],  # Regularization parameter
    'max_iter': [250, 500, 1000, 2000, 3000],  # Maximum number of iterations
    'tol': [1e-5, 1e-4, 1e-3, 1e-2],  # Tolerance for stopping criteria
}

# Initialize GridSearchCV with cross-validation
grid_search = GridSearchCV(estimator=model, param_grid=param_grid, cv=5, scoring='accuracy', n_jobs=-1) #f1

# Fit the grid search to the training data
grid_search.fit(X_train, y_train[target])

# Use the best model from the grid search to make predictions on the test set
best_model = grid_search.best_estimator_
lsvc_model = best_model
y_pred = best_model.predict(X_test)

cr = classification_report(y_test[target], y_pred, output_dict=True)
score_dict["linearsvc"] = {
    "precision":round(cr["1.0"]["precision"], 2),
    "recall":round(cr["1.0"]["recall"], 2),
    "f1-score":round(cr["1.0"]["f1-score"], 2),
    "accuracy":round(cr["accuracy"], 2)
}
score_dict["linearsvc"]

{'precision': 0.63, 'recall': 0.77, 'f1-score': 0.69, 'accuracy': 0.67}

In [93]:
# svm = LinearSVC()
# clf = CalibratedClassifierCV(svm) 
# clf.fit(X_train, y_train)
# y_proba = clf.predict_proba(X_test)

In [94]:
best_model.get_params()

{'C': 10.0,
 'class_weight': None,
 'dual': 'auto',
 'fit_intercept': True,
 'intercept_scaling': 1,
 'loss': 'squared_hinge',
 'max_iter': 250,
 'multi_class': 'ovr',
 'penalty': 'l2',
 'random_state': 0,
 'tol': 0.001,
 'verbose': 0}

In [95]:
lsvc_coef_df = get_coefficient_df(lsvc_model, x_columns)
print(lsvc_coef_df.head(18))  # Show top 10 features by importance

                        Feature  Coefficient
13     x_ll_memb_soe_Dahyun_mnh     1.694784
14      x_ll_memb_soe_Jihyo_btd    -1.255923
0       x_xs_beha_evt_live_tour     1.088208
16       x_ll_memb_soe_Sana_loa    -0.878770
19     x_ll_memb_soe_Nayeon_sgl     0.862350
1      x_xs_demo_regi_group_num     0.612704
6           x_xs_comp_west_sent     0.468905
27     x_xs_demo_heal_group_num     0.448682
18      x_ll_memb_soe_Jihyo_pds    -0.420945
23     x_xs_comp_west_sent_japn     0.415974
5       x_xs_demo_gen_group_num    -0.414108
21     x_ll_memb_soe_Dahyun_knk     0.408091
17     x_ll_memb_soe_Tzuyu_none    -0.383705
22  x_ll_memb_soe_Chaeyoung_csm     0.374771
2        x_xs_beha_mf_group_num     0.372413
3          x_xs_artm_mpp_memb_n     0.359884
15      x_ll_memb_soe_Jihyo_loa     0.356396
20   x_mm_demo_regi_america_c_s    -0.352569


In [96]:
print(lsvc_coef_df.tail(15))  # Show top 10 features by importance

                        Feature  Coefficient
22  x_ll_memb_soe_Chaeyoung_csm     0.374771
2        x_xs_beha_mf_group_num     0.372413
3          x_xs_artm_mpp_memb_n     0.359884
15      x_ll_memb_soe_Jihyo_loa     0.356396
20   x_mm_demo_regi_america_c_s    -0.352569
4            x_xs_comp_dis_sent     0.330462
25             x_xs_memb_pca_a1    -0.264155
24             x_xs_altm_pca_a1     0.250741
26             x_xs_artm_pca_a1    -0.238547
11          x_ss_demo_sex_other    -0.177402
12           x_mm_altm_jglt_dy6     0.136501
10      x_xs_demo_role_employed     0.057667
8      x_xs_demo_vert_group_num    -0.048559
9        x_xs_demo_role_student    -0.047886
7    x_ss_artm_mpp_memb_n_bit_0    -0.025716


## Generate KMean Clustering Model

In [97]:
# from sklearn.neighbors import KNeighborsClassifier
# from sklearn.model_selection import GridSearchCV
# from sklearn.metrics import classification_report

# # Initialize the KNN model
# model = KNeighborsClassifier()

# # Hyperparameter grid to search over
# param_grid = {
#     'n_neighbors': [3, 5, 7, 9, 11, 13, 15],  # Number of neighbors
#     'weights': ['uniform', 'distance'],  # Weight function used in prediction
#     'algorithm': ['auto', 'ball_tree', 'kd_tree', 'brute'],  # Algorithm to compute nearest neighbors
#     'leaf_size': [10, 20, 30, 40, 50],  # Leaf size for BallTree or KDTree
#     'p': [1, 2],  # Power parameter for Minkowski metric (1: Manhattan, 2: Euclidean)
# }

# # Initialize GridSearchCV with cross-validation
# grid_search = GridSearchCV(
#     estimator=model,
#     param_grid=param_grid,
#     cv=5,
#     scoring='accuracy',
#     n_jobs=-1
# )

# # Fit the grid search to the training data
# grid_search.fit(X_train, y_train["y_album_buying"])

# # Get the best parameters
# print("Best parameters found: ", grid_search.best_params_)

# # Use the best model from the grid search to make predictions on the test set
# best_model = grid_search.best_estimator_
# knn_model = best_model
# y_pred = best_model.predict(X_test)

# # Generate classification report
# cr = classification_report(y_test["y_album_buying"], y_pred, output_dict=True)

# # Store metrics in score dictionary
# score_dict["knn"] = {
#     "precision": round(cr["1.0"]["precision"], 2),
#     "recall": round(cr["1.0"]["recall"], 2),
#     "f1-score": round(cr["1.0"]["f1-score"], 2),
#     "accuracy": round(cr["accuracy"], 2)
# }
# score_dict["knn"]

In [98]:
#knn_model.get_params()

## Generate Ensamble Model

In [99]:
from sklearn.ensemble import VotingClassifier

# Create the voting classifier with your existing models
ensemble = VotingClassifier(
    estimators=[('rf', rf_model), ('log', log_model), ('svc', lsvc_model)], #knn_model
    #estimators=[('rf', rf_model), ('knn', knn_model), ('svc', lsvc_model)], #knn_model
    voting='hard'  # Use 'soft' for weighted voting if all models support predict_proba
)

# Fit and predict
ensemble.fit(X_train, y_train[target])
y_pred = ensemble.predict(X_test)

cr = classification_report(y_test[target], y_pred, output_dict=True)
score_dict["ensamble_mod"] = {
   "precision": round(cr["1.0"]["precision"], 2),
   "recall": round(cr["1.0"]["recall"], 2),
   "f1-score": round(cr["1.0"]["f1-score"], 2),
   "accuracy": round(cr["accuracy"], 2),
   #"best_params": grid_search.best_params_
}
score_dict["ensamble_mod"]

{'precision': 0.63, 'recall': 0.77, 'f1-score': 0.69, 'accuracy': 0.67}

# Compare Scores

In [100]:
model_report_df = pd.DataFrame(score_dict).T.rename_axis('model').reset_index().drop("n_features_used", axis=1)  # T to transpose and get models as rows

df_melted = model_report_df.melt(id_vars=['model'], var_name='score_type', value_name='score')

#df_melted
# fig = px.scatter(
#     x=df_melted["score"],
#     y=df_melted["score_type"],
#     color=df_melted["model"]
#     )
# fig.show()

fig = px.scatter(
    x=model_report_df["f1-score"],
    y=model_report_df["accuracy"],
    color=model_report_df["model"]
    )
fig.update_layout(
    xaxis_title="F1-Score",
    yaxis_title="Accuracy"
)
fig.show()

In [101]:
len(X_test.columns)

28

In [102]:
model_report_df[["model", "f1-score", "accuracy"]]

,model,f1-score,accuracy
0,baseline_rand,0.50,0.50
1,baseline_crt,0.51,0.50
2,baseline_100,0.66,0.49
3,model_logistic,0.69,0.67
4,logistic_l1,0.71,0.70
5,decision_tree,0.70,0.66
6,random_forest,0.73,0.71
7,linearsvc,0.69,0.67
8,ensamble_mod,0.69,0.67


In [103]:
# Only Top Features
## Best F1 Score (0.71) (model logistic & linear svc)
## Best Accuracy (0.67) (model logistic & linear svc)

# Only Top Features + 4
## Best F1 Score (0.72) (model logistic & linear svc)
## Best Accuracy (0.69) (linear svc)

# Only Top Features + 4 + pca
## Best F1 Score (0.70) (model logistic, logistic_l1,  linear svc)
## Best Accuracy (0.68) (model logistic, logistic_l1,  linear svc)


# Only Top Features + 4 + clustering
## Best F1 Score (0.72) (model logistic, logistic_l1,  linear svc)
## Best Accuracy (0.68) (model logistic, logistic_l1,  linear svc)

# Only Top B Features
## Best F1 Score (0.66) (model logistic)
## Best Accuracy (0.64) (model logistic)

# Only Middle Features
## Best F1 Score (0.65) (random forest)
## Best Accuracy Score (0.59) (random forest & model_logistic & svm_rbf)

# Without Bottom Features
## Best F1 Score (0.67) (random forest)
## Best Accuracy Score (0.64) (random forest)

model	precision	recall	f1-score	accuracy
0	baseline_50	0.52	0.50	0.51	0.50
1	baseline_52	0.52	0.52	0.52	0.50
2	baseline_100	0.52	1.00	0.68	0.52
3	model_logistic	0.52	0.59	0.55	0.53
4	logistic_l1	0.52	0.59	0.55	0.53
5	decision_tree	0.49	0.72	0.58	0.50
6	random_forest	0.50	0.52	0.51	0.50
7	svm_rbf	0.52	0.71	0.60	0.53
8	linearsvc	0.52	0.58	0.55	0.53

In [104]:
{'baseline_50': {'precision': 0.53,
  'recall': 0.53,
  'f1-score': 0.53,
  'accuracy': 0.54},
 'baseline_52': {'precision': 0.51,
  'recall': 0.54,
  'f1-score': 0.52,
  'accuracy': 0.52},
 'model_logistic': {'precision': 0.61,
  'recall': 0.68,
  'f1-score': 0.65,
  'accuracy': 0.63},
 'logistic_l1': {'precision': 0.61,
  'recall': 0.81,
  'f1-score': 0.7,
  'accuracy': 0.65,
  'n_features_used': 20},
 'decision_tree': {'precision': 0.59,
  'recall': 0.61,
  'f1-score': 0.6,
  'accuracy': 0.6},
 'random_forest': {'precision': 0.61,
  'recall': 0.75,
  'f1-score': 0.67,
  'accuracy': 0.64},
 'svm_rbf': {'precision': 0.56,
  'recall': 0.72,
  'f1-score': 0.63,
  'accuracy': 0.58},
 'linearsvc': {'precision': 0.64,
  'recall': 0.7,
  'f1-score': 0.67,
  'accuracy': 0.65}}

{'baseline_50': {'precision': 0.53,
  'recall': 0.53,
  'f1-score': 0.53,
  'accuracy': 0.54},
 'baseline_52': {'precision': 0.51,
  'recall': 0.54,
  'f1-score': 0.52,
  'accuracy': 0.52},
 'model_logistic': {'precision': 0.61,
  'recall': 0.68,
  'f1-score': 0.65,
  'accuracy': 0.63},
 'logistic_l1': {'precision': 0.61,
  'recall': 0.81,
  'f1-score': 0.7,
  'accuracy': 0.65,
  'n_features_used': 20},
 'decision_tree': {'precision': 0.59,
  'recall': 0.61,
  'f1-score': 0.6,
  'accuracy': 0.6},
 'random_forest': {'precision': 0.61,
  'recall': 0.75,
  'f1-score': 0.67,
  'accuracy': 0.64},
 'svm_rbf': {'precision': 0.56,
  'recall': 0.72,
  'f1-score': 0.63,
  'accuracy': 0.58},
 'linearsvc': {'precision': 0.64,
  'recall': 0.7,
  'f1-score': 0.67,
  'accuracy': 0.65}}

In [105]:
score_dict_old = {'baseline_50': {'precision': 0.57,
  'recall': 0.58,
  'f1-score': 0.58,
  'accuracy': 0.56},
 'baseline_52': {'precision': 0.56,
  'recall': 0.6,
  'f1-score': 0.58,
  'accuracy': 0.54},
 'model_logistic': {'precision': 0.63,
  'recall': 0.73,
  'f1-score': 0.68,
  'accuracy': 0.64},
 'decision_tree': {'precision': 0.58,
  'recall': 0.62,
  'f1-score': 0.6,
  'accuracy': 0.57},
 'linearsvc': {'precision': 0.64,
  'recall': 0.73,
  'f1-score': 0.68,
  'accuracy': 0.64}}

## Calculate Most Fequent Features

In [106]:
from collections import Counter

def print_frequencies(list1, list2, list3):
    # Combine all lists
    all_items = list1 + list2 + list3
    
    # Count frequencies
    frequencies = Counter(all_items)
    
    # Sort by frequency in descending order
    sorted_items = sorted(frequencies.items(), key=lambda x: x[1], reverse=True)
    
    # Print results
    for item, freq in sorted_items:
        print(f"{item}: {freq}")

# Example usage:




In [107]:
logistic_l1_t10 = logistic_l1_coef_df.head(10)["Feature"].to_list()
rf_t10 = list(rf_t20.keys())[:10]
lscv_t10 = lsvc_coef_df.head(10)["Feature"].to_list()

print("Top Attributes")
print_frequencies(logistic_l1_t10, rf_t10, lscv_t10)

Top Attributes
x_xs_beha_evt_live_tour: 3
x_xs_demo_regi_group_num: 3
x_xs_comp_west_sent: 3
x_xs_demo_heal_group_num: 3
x_ll_memb_soe_Dahyun_mnh: 2
x_ll_memb_soe_Jihyo_btd: 2
x_ll_memb_soe_Nayeon_sgl: 2
x_ll_memb_soe_Sana_loa: 2
x_xs_demo_gen_group_num: 1
x_ll_memb_soe_Dahyun_knk: 1
x_xs_memb_pca_a1: 1
x_xs_artm_pca_a1: 1
x_xs_altm_pca_a1: 1
x_xs_beha_mf_group_num: 1
x_xs_artm_mpp_memb_n: 1
x_xs_comp_dis_sent: 1
x_ll_memb_soe_Jihyo_pds: 1
x_xs_comp_west_sent_japn: 1


In [108]:
logistic_l1_b10 = logistic_l1_coef_df.tail(10)["Feature"].to_list()
rf_b10 = list(rf_b20.keys())[-10:]
lscv_b10 = lsvc_coef_df.tail(10)["Feature"].to_list()

print("Bottom Attributes")
print_frequencies(logistic_l1_b10, rf_b10, lscv_b10)

Bottom Attributes
x_ss_demo_sex_other: 3
x_xs_comp_dis_sent: 2
x_xs_memb_pca_a1: 2
x_xs_altm_pca_a1: 2
x_xs_artm_pca_a1: 2
x_mm_altm_jglt_dy6: 2
x_xs_demo_role_student: 2
x_xs_demo_vert_group_num: 2
x_xs_demo_role_employed: 2
x_ss_artm_mpp_memb_n_bit_0: 2
x_ll_memb_soe_Tzuyu_none: 1
x_ll_memb_soe_Dahyun_knk: 1
x_ll_memb_soe_Jihyo_loa: 1
x_ll_memb_soe_Jihyo_btd: 1
x_ll_memb_soe_Nayeon_sgl: 1
x_ll_memb_soe_Jihyo_pds: 1
x_xs_comp_west_sent_japn: 1
x_ll_memb_soe_Sana_loa: 1
x_ll_memb_soe_Dahyun_mnh: 1
